In [1]:
import pandas as pd
import numpy as np

def read_excel(file_name):
    df = pd.read_excel(file_name)
    return df

def read_txt(file_name):
    file = open(file_name)
    lines = file.readlines()
    return(lines[0])

In [2]:
import os
import glob

def get_files(subfolder, extension):
    dir = f"{os.getcwd()}/content/{subfolder}/"
    tables = glob.glob(f"{dir}*.{extension}")
    return tables

In [3]:
class Analizer:
    def __init__(self, boundary):
        self.results = get_files(subfolder="results", extension="xlsx")
        self.results_df = pd.DataFrame()
        self.boundary = boundary
    
    def has_minimum_requirements(self, df, sort_by="r2"):
        sorted_df = df.sort_values(by=sort_by, ascending=False)
        top_r2 = sorted_df.head(1)[sort_by].values[0]
        if top_r2 < self.boundary:
            return False
        return True
    
    def concatenate_df(self, df, architecture):
        if self.has_minimum_requirements(df):
            df['Architecture'] = architecture
            df = df.rename(columns={'Unnamed: 0': 'model'})
            self.results_df = pd.concat([self.results_df, df], ignore_index=True) 

    def create_results_df(self):
        for file in self.results:
            df = read_excel(file)
            architecture = read_txt(file.replace(".xlsx", ".txt"))
            self.concatenate_df(df, architecture)
        self.results_df = self.results_df.sort_values(by="r2", ascending=False, ignore_index=True)

    def discard_below_average(self, sort_by):
        column_mean = self.results_df[sort_by].mean()      
        self.results_df = self.results_df[self.results_df[sort_by] >= column_mean]
    
    def discard_high_standard_deviation(self):
        r2_val, r2_test = self.results_df['r2_val'], self.results_df['r2_test']
        std_devs = np.abs(r2_val - r2_test)
        mean_std_dev = std_devs.mean()
        self.results_df = self.results_df[std_devs < mean_std_dev]

    def clean_folder(self, subfolder, extension, remove_last=True):
        files = get_files(subfolder, extension)
        models = self.results_df["model"]
        if (remove_last):
            models = models.apply(lambda x: '_'.join(x.rsplit('_', 1)[:-1]))
        for file in files:
            file_name = os.path.basename(file).split('.')[0]
            file_parts = file_name.split('_')            
            dataset_model = f"model_{file_parts[1]}_{file_parts[2]}" 
            if (remove_last == False):
                dataset_model = (f"{dataset_model}_{file_parts[3]}")
            if dataset_model not in models.values:
                os.remove(file)   
        
    def Analize(self):
        self.create_results_df()
        self.discard_below_average(sort_by="r2")
        self.discard_below_average(sort_by="r2_vt")
        self.discard_high_standard_deviation()
        self.results_df.to_excel(f"better_results.xlsx", index=True)
        display(self.results_df)


In [5]:
analize = Analizer(0.9)
analize.Analize()
analize.clean_folder(subfolder="dataset", extension="pkl")
analize.clean_folder(subfolder="results", extension="xlsx")
analize.clean_folder(subfolder="results", extension="txt")
analize.clean_folder(subfolder="models", extension="keras", remove_last=False)



,model,r2,r2_sup,r2_test,r2_val,r2_vt,mse,mse_sup,mse_test,mse_val,mse_vt,mape,rmse,r2_adj,rsd,aic,bic,Architecture
0,model_13_9_24,0.999967,0.620445,0.999911,0.999928,0.999936,0.000019,0.225321,0.000097,0.000013,0.000055,0.002035,0.004400,1.000060,0.004588,95.704219,140.802624,"Hidden Size=[4, 4], regularizer=0.5, learning_..."
1,model_13_9_23,0.999967,0.620327,0.999912,0.999933,0.999937,0.000019,0.225390,0.000096,0.000012,0.000054,0.002052,0.004405,1.000060,0.004592,95.700294,140.798699,"Hidden Size=[4, 4], regularizer=0.5, learning_..."
2,model_13_9_22,0.999967,0.620186,0.999913,0.999939,0.999939,0.000020,0.225474,0.000094,0.000011,0.000053,0.002071,0.004419,1.000061,0.004607,95.687250,140.785655,"Hidden Size=[4, 4], regularizer=0.5, learning_..."
3,model_13_9_21,0.999967,0.620027,0.999914,0.999943,0.999940,0.000020,0.225568,0.000093,0.000011,0.000052,0.002093,0.004450,1.000062,0.004639,95.659455,140.757861,"Hidden Size=[4, 4], regularizer=0.5, learning_..."
4,model_13_9_20,0.999966,0.619824,0.999916,0.999946,0.999941,0.000020,0.225689,0.000092,0.000010,0.000051,0.002115,0.004499,1.000063,0.004690,95.615925,140.714330,"Hidden Size=[4, 4], regularizer=0.5, learning_..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1259,model_13_4_1,0.998770,0.609001,0.999977,0.999981,0.999980,0.000730,0.232114,0.000015,0.000017,0.000016,0.015713,0.027026,1.002272,0.028177,88.443758,133.542164,"Hidden Size=[4, 4], regularizer=0.5, learning_..."
1287,model_11_1_16,0.998572,0.625625,0.998987,0.999832,0.999348,0.000848,0.222245,0.000978,0.000030,0.000504,0.011806,0.029117,1.002637,0.030356,88.145763,133.244168,"Hidden Size=[5, 3], regularizer=0.3, learning_..."
1297,model_13_4_0,0.998488,0.608070,0.999994,0.999990,0.999992,0.000898,0.232666,0.000004,0.000009,0.000006,0.016942,0.029960,1.002791,0.031236,88.031528,133.129934,"Hidden Size=[4, 4], regularizer=0.5, learning_..."
1320,model_11_1_15,0.998274,0.630204,0.998897,0.999832,0.999291,0.001025,0.219527,0.001064,0.000030,0.000547,0.012768,0.032012,1.003187,0.033375,87.766582,132.864988,"Hidden Size=[5, 3], regularizer=0.3, learning_..."
